# ISyE 524 installation check

Run this notebook from top to bottom after setting up the course environment. If every cell succeeds, Julia, JuMP, HiGHS, formatted output, tables, plots, and Markdown mathematics are working.

In VS Code, use **Run All** from the notebook toolbar. If prompted for a kernel, select **Julia**.

## Markdown mathematics

Inline mathematics should render here: $x \geq 0$ and $y \geq 0$.

The installation check solves

$$
\begin{aligned}
\max_{x,y}\quad & 3x + 4y \\
\text{s.t.}\quad & 2x + y \leq 10, \\
& x + 3y \leq 12, \\
& x,y \geq 0.
\end{aligned}
$$

The optimal solution is $(x,y)=(3.6,2.8)$ with objective value $22$.

In [ ]:
using Pkg

if VERSION.major != 1 || VERSION.minor != 12
    error(
        "This course requires Julia 1.12.x, but this kernel is Julia $(VERSION).",
    )
end

required_packages = ["JuMP", "HiGHS", "DataFrames", "Plots", "Printf"]
missing_packages = setdiff(
    required_packages,
    collect(keys(Pkg.project().dependencies)),
)
isempty(missing_packages) || error(
    "The course environment is not active. Missing: $(join(missing_packages, ", "))",
)

println("Julia version: ", VERSION)
println("Active project: ", Base.active_project())
println("Course environment detected.")

In [ ]:
using JuMP, HiGHS, Printf
import MathOptInterface as MOI

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x >= 0)
@variable(model, y >= 0)
@constraint(model, 2x + y <= 10)
@constraint(model, x + 3y <= 12)
@objective(model, Max, 3x + 4y)

optimize!(model)
termination_status(model) == MOI.OPTIMAL ||
    error("HiGHS did not solve the model.")

@printf("x = %.1f\n", value(x))
@printf("y = %.1f\n", value(y))
@printf("objective = %.1f\n", objective_value(model))

In [ ]:
using DataFrames

DataFrame(
    variable = ["x", "y"],
    value = [value(x), value(y)],
)

In [ ]:
using Plots

figure = plot(
    Shape([0.0, 5.0, 3.6, 0.0], [0.0, 0.0, 2.8, 4.0]);
    alpha = 0.25,
    aspect_ratio = :equal,
    xlabel = "x",
    ylabel = "y",
    label = "feasible region",
    title = "Installation-check linear program",
)
scatter!(figure, [value(x)], [value(y)]; label = "optimum")

## Success

If you see the Julia version and project path, the solution $(3.6,2.8)$, a two-row table, and a plot with the optimum, your installation is ready for the course.

You can also run **ISyE 524: Run environment check** from VS Code's task list for a command-line verification.